[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc5_abtest/exercices/seance1_exercices.ipynb)

# Séance 5.1 — Causalité et A/B testing — mesurer ce qu'une campagne fait vraiment

**Exercices** · durée : 6h (2h de cours, 2h d'étude de cas, 2h de correction)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer une question prédictive d'une question causale
- nommer le contrefactuel, l'ATE et l'ATT, et dire pourquoi on ne les observe jamais
- décomposer une comparaison de moyennes en effet causal + biais de sélection
- vérifier qu'un tirage au sort a fonctionné avec un tableau d'équilibre
- chiffrer l'effet d'un A/B test, son incertitude, et le traduire en décision

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
data = pd.read_csv(BASE + "hillstrom.csv")
AUCUN, HOMME, FEMME = "No E-Mail", "Mens E-Mail", "Womens E-Mail"


def comparer(a, b):
    """Effet, intervalle a 95 % et p-value entre deux groupes."""
    effet = a.mean() - b.mean()
    es = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    return pd.Series({"effet": effet, "bas_95": effet - 1.96 * es,
                      "haut_95": effet + 1.96 * es, "p_value": p})


print(data.shape, "|", data["segment"].unique())

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Première inspection

> **Votre mission :**
> - Combien de clients le fichier contient-il ? → `n_lignes`
> - Combien de valeurs manquantes en tout ? → `n_manquants`
> - Combien de clients ont reçu l'email hommes ? → `n_homme`

In [ ]:
n_lignes = data.____[0]
n_manquants = data.isna().sum().____()
n_homme = data["segment"].value_counts()[HOMME]

print(n_lignes, "clients |", n_manquants, "valeurs manquantes |", n_homme, "traites")

In [ ]:
verifier("1a - nombre de clients", n_lignes == 64000, "data.shape[0]")
verifier("1b - valeurs manquantes", n_manquants == 0, "sommez deux fois")
verifier("1c - clients traites", n_homme == 21307, "value_counts() sur la colonne segment")

### Exercice 2 — Un monde sans tirage au sort

> **Votre mission :**
> - Avant d'analyser la vraie expérience, simulons le monde où l'entreprise **n'aurait pas** randomisé : comme beaucoup, elle aurait écrit à ses **meilleurs clients**.
> - On garde les clients « email hommes » dont la dépense passée dépasse la médiane, et les clients « aucun email » en dessous.
> - Compléter la médiane, puis compter les clients de cette base → `n_monde`

In [ ]:
mediane = data["history"].____()

monde = pd.concat([
    data.query("segment == @HOMME and history > @mediane"),
    data.query("segment == @AUCUN and history <= @mediane"),
])
monde["email"] = (monde["segment"] == HOMME).astype(int)

n_monde = len(monde)
print("mediane :", mediane, "| base simulee :", n_monde, "clients")

In [ ]:
verifier("2 - taille du monde parallele", n_monde == 21333,
         "la mediane coupe l'echantillon en deux, pas le quartile")

### Exercice 3 — L'estimation naïve

> **Votre mission :**
> - Dans ce monde parallèle, calculer la dépense moyenne des clients avec email et sans email, puis leur différence → `naif` (arrondie à 2 décimales).
> - **Notez ce chiffre quelque part.** C'est ce qu'un analyste pressé appellerait « l'effet de l'email ». Nous le confronterons à la vérité à l'exercice 8.

In [ ]:
moyennes = monde.groupby("email")["spend"].____()
naif = round(moyennes[1] - moyennes[0], 2)

print(moyennes.round(3))
print("estimation naive de « l'effet de l'email » :", naif, "$")

In [ ]:
verifier("3 - estimation naive", naif == 1.32,
         "moyenne des email=1 moins moyenne des email=0, arrondie a 2 decimales")

### Exercice 4 — Diagnostiquer le biais

> **Votre mission :**
> - Toujours dans le monde parallèle : comparer la **dépense passée** (`history`) des deux groupes → `passe_email` et `passe_sans` (arrondies à 1 décimale).
> - Ces deux clients auraient-ils dépensé pareil **sans aucun email** ?

In [ ]:
passe = monde.groupby("email")["history"].____().round(1)

passe_email = passe[1]
passe_sans = passe[0]
print("depense passee :", passe_email, "$ avec email contre", passe_sans, "$ sans")

In [ ]:
verifier("4a - depense passee, groupe traite", passe_email == 414.0, "arrondissez a 1 decimale")
verifier("4b - depense passee, groupe temoin", passe_sans == 73.7, "groupby sur email, colonne history")

### Exercice 5 — Le test d'équilibre

> **Votre mission :**
> - Retour à la **vraie** expérience. Si le tirage au sort a fonctionné, les variables mesurées **avant** l'envoi doivent être quasi identiques d'un groupe à l'autre.
> - Construire le tableau d'équilibre, puis mesurer l'écart maximal de dépense passée entre les trois groupes → `ecart_passe` (arrondi à 2 décimales).

In [ ]:
equilibre = data.groupby("segment").agg(
    clients=("segment", "size"),
    recence=("recency", "____"),
    passe=("history", "mean"),
    nouveaux=("newbie", "mean"),
)
ecart_passe = round(equilibre["passe"].max() - equilibre["passe"].min(), 2)

print("ecart maximal sur la depense passee :", ecart_passe, "$")
equilibre.round(3)

In [ ]:
verifier("5 - ecart sur la depense passee", ecart_passe == 1.95,
         "le maximum de la colonne passe moins son minimum")

### Exercice 6 — Les résultats de l'expérience

> **Votre mission :**
> - Pour chaque groupe : le nombre de clients, le taux de visite, le taux d'achat et la dépense moyenne.
> - Récupérer le taux d'achat du groupe « email hommes » → `achat_homme` (arrondi à 4 décimales).
> - *Rappel :* pour une variable qui vaut 0 ou 1, la **moyenne est la proportion de 1**.

In [ ]:
resultats = data.groupby("segment").agg(
    clients=("segment", "size"),
    visite=("visit", "mean"),
    achat=("conversion", "____"),
    depense=("spend", "mean"),
)
achat_homme = round(resultats.loc[HOMME, "achat"], 4)

print("taux d'achat, email hommes :", achat_homme)
resultats.round(4)

In [ ]:
verifier("6 - taux d'achat du groupe traite", achat_homme == 0.0125,
         "la moyenne d'une colonne 0/1 est sa proportion de 1")

### Exercice 7 — Effet absolu, effet relatif

> **Votre mission :**
> - Mesurer l'effet de l'email hommes sur le taux d'achat, par rapport au groupe sans email.
> - En **points de pourcentage** → `abs_conv` (2 décimales), et en **pourcentage du niveau de départ** → `rel_conv` (1 décimale).
> - Les deux décrivent le même résultat. Laquelle des deux mettriez-vous sur une slide ?

In [ ]:
ecart_achat = resultats.loc[HOMME, "achat"] - resultats.loc[____, "achat"]

abs_conv = round(100 * ecart_achat, 2)
rel_conv = round(100 * ecart_achat / resultats.loc[AUCUN, "achat"], 1)

print("effet absolu :", abs_conv, "points | effet relatif :", rel_conv, "%")

In [ ]:
verifier("7a - effet absolu", abs_conv == 0.68, "en points : multipliez l'ecart par 100")
verifier("7b - effet relatif", rel_conv == 118.8,
         "divisez l'ecart par le taux du groupe temoin, pas par 1")

### Exercice 8 — Le moment de vérité

> **Votre mission :**
> - Mesurer l'effet **réel** de l'email hommes sur la dépense → `effet_dep` (2 décimales).
> - Le comparer à votre estimation naïve de l'exercice 3 : de combien le monde parallèle se trompait-il ? → `biais` (2 décimales)

In [ ]:
effet_dep = round(resultats.loc[HOMME, "depense"] - resultats.loc[AUCUN, "depense"], 2)
biais = round(naif - ____, 2)

print("effet reel (randomise) :", effet_dep, "$")
print("estimation naive (ex. 3) :", naif, "$")
print("biais de selection :", biais, "$")

In [ ]:
verifier("8a - effet reel sur la depense", effet_dep == 0.77,
         "difference des colonnes depense entre HOMME et AUCUN")
verifier("8b - taille du biais", biais == 0.55, "l'estimation naive moins l'effet reel")

### Exercice 9 — Est-ce que ça peut être le hasard ?

> **Votre mission :**
> - Même avec un tirage au sort parfait, deux groupes ne sont jamais exactement identiques. L'effet mesuré dépasse-t-il ce que le hasard seul produirait ?
> - Utiliser `comparer(...)` sur la dépense, email hommes contre aucun email.
> - Relever les bornes de l'intervalle à 95 % → `bas` et `haut` (2 décimales).

In [ ]:
dep_homme = data.query("segment == @HOMME")["spend"]
dep_aucun = data.query("segment == @AUCUN")["spend"]

test_dep = comparer(dep_homme, ____)
bas = round(test_dep["bas_95"], 2)
haut = round(test_dep["haut_95"], 2)

print("intervalle a 95 % : de", bas, "a", haut, "$")
test_dep.round(4)

In [ ]:
verifier("9a - borne basse", bas == 0.49, "test_dep['bas_95'], arrondi a 2 decimales")
verifier("9b - borne haute", haut == 1.05, "test_dep['haut_95']")
verifier("9c - l'intervalle exclut zero", bas > 0, "regardez le signe de la borne basse")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Calculer l'effet relatif de l'email hommes sur la **dépense** → `rel_dep` (1 décimale).
> - Puis rédigez en commentaire, en trois phrases maximum, ce que vous diriez à la directrice marketing.

In [ ]:
rel_dep = round(100 * effet_dep / resultats.loc[____, "depense"], 1)

print("effet relatif sur la depense :", rel_dep, "%")

# Votre recommandation :

In [ ]:
verifier("10 - effet relatif sur la depense", rel_dep == 118.0,
         "l'effet divise par la depense moyenne du groupe temoin")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 11 — Traduire dans le langage du cours

> **Votre mission :**
> - Répondez en une phrase par question, sans coder.
> - 1. Ici, qu'est-ce que $T_i$ ? (Attention : il y a **deux** traitements possibles — on comparera chacun au groupe sans email.)
> - 2. Si on s'intéresse aux dépenses, qu'est-ce que $Y_i$ ?
> - 3. Le client n° 42 a reçu l'email hommes et a dépensé 0 $. Que représenterait son $Y_{0,42}$ ? Peut-on l'observer ?
> - 4. Comment s'appelle, dans le cours, ce résultat qu'on ne peut jamais observer ?

**Vos réponses :**

1. 
2. 
3. 
4. 

### Question 12 — Prédiction ou causalité ?

> **Votre mission :**
> - Pour chaque question, dire s'il s'agit d'une question **prédictive** (bloc 4) ou **causale** (ce bloc), et justifier en une phrase.
> - 1. Quels clients ont le plus de chances d'acheter dans les deux prochaines semaines ?
> - 2. Envoyer un email augmente-t-il la probabilité d'achat ?
> - 3. Combien ce client va-t-il probablement dépenser ?
> - 4. Que dépenseraient nos clients si nous leur envoyions l'email hommes plutôt que rien ?

**Vos réponses :**

1. 
2. 
3. 
4. 

### Question 13 — Le détecteur de randomisation cassée

> **Votre mission :**
> - Refaire le tableau d'équilibre de l'exercice 5, mais sur `monde` (en groupant par `email`).
> - Qu'est-ce qui saute aux yeux ? Si on vous avait livré cette base en vous affirmant qu'elle était randomisée, ce test vous aurait-il sauvé ?

### Question 14 — Trois graphiques, trois indicateurs

> **Votre mission :**
> - Faire un graphique en barres horizontales par indicateur, à partir du tableau `resultats` : taux de visite, taux d'achat, dépense moyenne.
> - Trier avant de tracer, et une idée par figure.
> - *Rappel :* `resultats["visite"].sort_values().plot(kind="barh", figsize=(7, 3))`

### Question 15 — Quelle campagne est la meilleure ?

> **Votre mission :**
> - Attention au raccourci : « A bat le témoin, B bat le témoin, et l'effet de A est plus grand, donc A bat B ». Pour affirmer que A bat B, il faut **comparer A à B directement**.
> - Comparer l'email hommes à l'email femmes sur le taux d'achat, puis sur la dépense.
> - La supériorité de l'un sur l'autre est-elle établie dans les **deux** cas ?

### Question 16 — Le petit concurrent

> **Votre mission :**
> - Un concurrent plus petit n'a que **2 000 clients** pour mener le même test. Simuler sa situation avec `data.sample(2000, random_state=0)`, puis comparer l'email hommes au groupe sans email sur le taux d'achat.
> - Recommencer avec plusieurs valeurs de `random_state`. Que constatez-vous sur la largeur de l'intervalle et sur la stabilité de l'effet estimé ?
> - L'effet de l'email a-t-il « disparu » chez lui ? Que peut-il conclure — et surtout, que ne peut-il **pas** conclure ?

### Question 17 — Est-ce que ça rapporte ?

> **Votre mission :**
> - La directrice donne ses paramètres : l'entreprise peut contacter **100 000 clients**, la marge est de **40 %** du chiffre d'affaires, et chaque email coûte **0,05 $**.
> - Bénéfice net par client = effet sur la dépense × taux de marge − coût de l'email.
> - Calculer le bénéfice attendu sur 100 000 clients pour chaque campagne. Puis refaire le calcul avec les **bornes** de l'intervalle de confiance : la décision tient-elle dans le scénario le plus prudent ?

### Question 18 — Segmenter, et se méfier de soi-même

> **Votre mission :**
> - Les campagnes ont-elles le même effet selon le canal d'achat habituel (`channel`) ? Calculer la dépense moyenne par `channel` et par `segment`, puis l'effet de chaque campagne dans chaque canal.
> - Repérer le canal où l'écart semble le plus fort. Recommanderiez-vous de concentrer le budget dessus ?
> - *Nouveau :* `data.pivot_table(values="spend", index="channel", columns="segment", aggfunc="mean")` croise deux variables en un tableau.

### Question 19 — Le stagiaire revient

> **Votre mission :**
> - Nouvelle idée du stagiaire : « Comparons la dépense des clients qui ont **visité** le site à celle des autres : on verra l'effet causal de la visite ! »
> - Reproduire son calcul : la dépense moyenne des clients traités **qui ont visité**, contre celle du groupe témoin entier. Retrouvez-vous les chiffres de sa slide ?
> - Regarder aussi ce que dépensent les clients traités qui n'ont **pas** visité.
> - `visit` est-elle une variable mesurée **avant** ou **après** l'envoi ? Pourquoi cette comparaison retombe-t-elle exactement dans le piège du monde parallèle ?

### Question 20 — La note à la direction

> **Votre mission :**
> - Rédigez votre recommandation en **cinq phrases maximum**. Elle sera jugée sur cette liste :
> - une stratégie claire est recommandée ; l'effet sur l'achat et sur la dépense est cité, en choisissant honnêtement entre absolu et relatif ; l'incertitude est mentionnée (un intervalle, pas seulement « significatif ») ; l'ordre de grandeur du bénéfice apparaît ; au moins une limite est signalée ; **zéro jargon** — la directrice n'a jamais entendu parler de p-value.
> - Puis, en deux lignes chacune : l'expérience date de 2008, sur un site américain — peut-on en garantir les résultats en France en 2026, et comment s'appelle ce problème ? Et : avant de lancer un nouveau test, faut-il fixer le KPI, les segments et la durée à l'avance, ou peut-on attendre les résultats pour choisir ?

**Votre note à la direction :**

...

**Validité dans le temps :**

...

**Ce qu'il faut fixer à l'avance :**

...